In [1]:
from datetime import date
import sys
sys.path.append('/home/potzschf/repos/')
from helperToolz.helpsters import *
from helperToolz.dicts_and_lists import *
from FieldWaterUseTools.FuncBox.Misc import warp_np_to_reference

In [ ]:
masterPath = '/data/Aldhani/eoagritwin/force/output/GOHAR/2026/'
tiles = dirfinder(masterPath)
template = '/data/Aldhani/eoagritwin/force/output/GOHAR/s2stm_2024_061-182.tif'
cubeName = 's2stm_2025_061-243'

In [3]:
bands = ["GREEN", "RED", "NIR", "REDEDGE1", "REDEDGE2", "REDEDGE3", "SWIR1", "SWIR2", "NDMI", "NDVI", "TC-BRIGHT", "TC-GREEN", "TC-WET"]

In [4]:
allFiles = getFilelist(f"{masterPath}", ".tif",deep=True)

In [ ]:
for band in bands:
    vrt_path = f"{masterPath}vrt/VRT_{band}_STM.vrt"
    sub_files = [file for file in allFiles if f"_{band}_STM" in file]
    vrt = gdal.BuildVRT(path_safe(vrt_path), sub_files, separate = False)
    vrt = None

    convertVRTpathsTOrelative(vrt_path)

    arr = stackReader(vrt_path)
    warp_np_to_reference(arr,vrt_path, template,resamp=gdal.GRA_NearestNeighbour,output_path=path_safe(f"{masterPath}warped/{band}.tif"), band_names=getBandNames(sub_files[0]))


In [7]:
# get cutted files files
bandNamesL = []
arrL = []
for file in getFilelist(f"{masterPath}warped/", ".tif"):
    bnames = getBandNames(file)
    bnamesL = [f"{file.split('/')[-1].split('.tif')[0]}_{bname}" for bname in bnames]
    for bnL in bnamesL:
        bandNamesL.append(bnL)

    arr = stackReader(file)
    for aa in range(arr.shape[-1]):
        arrL.append(arr[:,:,aa])

In [27]:
newNames = ['GREEN', 'NDMI', 'NDVI',  'NIR', 'REDEDGE1', 'REDEDGE2', 'REDEDGE3', 'RED', 'SWIR1', 'SWIR2', 'TC-BRIGHT','TC-GREEN', 'TC-WET']# [bn.split('_')[0] for bn in bandNamesL]
oldNames = ["GRN", "NDM", "NDV", "NIR", "RE1", "RE2", "RE3", "RED", "SW1", "SW2", "TCB", "TCG", "TCW"]

lkpup = dict(zip(newNames, oldNames))



In [28]:
lkpup

{'GREEN': 'GRN',
 'NDMI': 'NDM',
 'NDVI': 'NDV',
 'NIR': 'NIR',
 'REDEDGE1': 'RE1',
 'REDEDGE2': 'RE2',
 'REDEDGE3': 'RE3',
 'RED': 'RED',
 'SWIR1': 'SW1',
 'SWIR2': 'SW2',
 'TC-BRIGHT': 'TCB',
 'TC-GREEN': 'TCG',
 'TC-WET': 'TCW'}

In [31]:
qq = bandNamesL

In [34]:
for idx2, q in enumerate(qq):
    print(f"{q} = {bandNamesL[idx2]}")

GRN_Q05 = GRN_Q05
GRN_Q95 = GRN_Q95
GRN_Q50 = GRN_Q50
GRN_STD = GRN_STD
GRN_RNG = GRN_RNG
GRN_IQR = GRN_IQR
NDM_Q05 = NDM_Q05
NDM_Q95 = NDM_Q95
NDM_Q50 = NDM_Q50
NDM_STD = NDM_STD
NDM_RNG = NDM_RNG
NDM_IQR = NDM_IQR
NDV_Q05 = NDV_Q05
NDV_Q95 = NDV_Q95
NDV_Q50 = NDV_Q50
NDV_STD = NDV_STD
NDV_RNG = NDV_RNG
NDV_IQR = NDV_IQR
NIR_Q05 = NIR_Q05
NIR_Q95 = NIR_Q95
NIR_Q50 = NIR_Q50
NIR_STD = NIR_STD
NIR_RNG = NIR_RNG
NIR_IQR = NIR_IQR
RED_Q05 = RED_Q05
RED_Q95 = RED_Q95
RED_Q50 = RED_Q50
RED_STD = RED_STD
RED_RNG = RED_RNG
RED_IQR = RED_IQR
RED_Q05 = RED_Q05
RED_Q95 = RED_Q95
RED_Q50 = RED_Q50
RED_STD = RED_STD
RED_RNG = RED_RNG
RED_IQR = RED_IQR
RED_Q05 = RED_Q05
RED_Q95 = RED_Q95
RED_Q50 = RED_Q50
RED_STD = RED_STD
RED_RNG = RED_RNG
RED_IQR = RED_IQR
RED_Q05 = RED_Q05
RED_Q95 = RED_Q95
RED_Q50 = RED_Q50
RED_STD = RED_STD
RED_RNG = RED_RNG
RED_IQR = RED_IQR
SW1_Q05 = SW1_Q05
SW1_Q95 = SW1_Q95
SW1_Q50 = SW1_Q50
SW1_STD = SW1_STD
SW1_RNG = SW1_RNG
SW1_IQR = SW1_IQR
SW2_Q05 = SW2_Q05
SW2_Q95 = 

In [32]:
for idx, bandName in enumerate(bandNamesL):
    for k, v in lkpup.items():
        if k in bandName:
            bandNamesL[idx] = f"{v}_{bandName.split('_')[-1]}"


In [35]:
cube = np.dstack(arrL)
npTOdisk(cube, file, f"{masterPath}{cubeName}.tif", bands=len(arrL), bandnames=bandNamesL, d_type=gdal.GDT_Int16)